# 06 – Pipeline Testing / Evaluation
Dieses Notebook führt eine reproduzierbare Evaluation über `tests/datasets/` aus und schreibt Ergebnisse nach `tests/outputs/`.
Es vergleicht Modellantworten mit Gold-Labels und protokolliert Kennzahlen und Artefakte.


In [1]:
import json
import os
import re
import csv
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from tqdm import tqdm
from neo4j import GraphDatabase

load_dotenv()

# -----------------------------
# Configuration
# -----------------------------
# Use absolute path to the top-level tests outputs folder
OUTPUTS_DIR = Path("/app/tests") / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Models / endpoints
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
GENERATOR_MODEL = os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")
JUDGE_MODEL = os.getenv("OLLAMA_JUDGE_MODEL") or GENERATOR_MODEL

# Evaluation controls
MAX_SAMPLES: Optional[int] = None
SHOW_PROGRESS = True
SAVE_COMPONENT_OUTPUTS = True
MAX_DEBUG_DOCS = 5

# Neo4j for Graph Cypher validation (EXPLAIN)
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
_neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

print("OLLAMA_URL:", OLLAMA_URL)
print("GENERATOR_MODEL:", GENERATOR_MODEL)
print("JUDGE_MODEL:", JUDGE_MODEL)


OLLAMA_URL: http://ollama:11434
GENERATOR_MODEL: qwen2.5:32b-instruct
JUDGE_MODEL: qwen2.5:32b-instruct


In [2]:

# -----------------------------
# Metrics + IO
# -----------------------------

def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def exact_match(pred: str, gold: str) -> bool:
    return normalize_text(pred) == normalize_text(gold)

def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    gold_counts = Counter(gold_tokens)
    common = sum((pred_counts & gold_counts).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def save_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def save_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: Optional[List[str]] = None) -> None:
    """Write rows to CSV deterministically (UTF-8)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        # Stable field order: union of keys in first-appearance order
        seen = []
        for r in rows:
            for k in r.keys():
                if k not in seen:
                    seen.append(k)
        fieldnames = seen
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, "") for k in fieldnames})


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


In [3]:

# -----------------------------
# Graph Cypher checks
# -----------------------------

_FORBIDDEN_CYPHER_TOKENS = [
    "create", "merge", "delete", "set", "call", "load csv", "apoc", "drop", "remove"
]

def cypher_policy_check(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    if not cypher:
        return False, "empty cypher"
    lower = cypher.lower()
    for tok in _FORBIDDEN_CYPHER_TOKENS:
        if tok in lower:
            return False, f"forbidden token: {tok}"
    return True, None

def cypher_explain_ok(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    if not cypher:
        return False, "empty cypher"
    try:
        with _neo4j_driver.session() as session:
            session.run("EXPLAIN " + cypher).consume()
        return True, None
    except Exception as exc:
        return False, str(exc)


In [4]:
# -----------------------------
# Faithfulness removed
# -----------------------------
# The project previously used a FaithfulnessEvaluator to compute a separate
# faithfulness score. Per request this has been removed to simplify the
# notebook: only exact match / F1 are computed now. Any judge-related
# components were deleted to avoid unintended interactions.


In [5]:

# -----------------------------
# Pipelines
# -----------------------------
from src.graph.graph_pipeline import build_pipeline as build_graph_pipeline
from src.base.base_pipeline import create_rag_pipeline as build_base_pipeline, run_rag_pipeline

graph_pipe = build_graph_pipeline()
base_pipe = build_base_pipeline(ollama_model=GENERATOR_MODEL, ollama_url=OLLAMA_URL)


In [6]:

# -----------------------------
# Debug runners: build judge contexts
# -----------------------------

def _safe_get(d: Any, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return cur if cur is not None else default

def _extract_base_answer(result: Dict[str, Any]) -> str:
    llm_out = result.get("llm", {})
    if isinstance(llm_out, dict):
        replies = llm_out.get("replies", [])
        if replies:
            return str(replies[0])
    return ""

def build_context_snippets_from_graph_analysis(analysis_json: Dict[str, Any], max_records: int = 20) -> List[str]:
    """Turn Organizer JSON into a list of evidence snippets for the judge.

    Goal: small, flat, human-readable strings (GPT-5-mini-friendly).
    """
    raw_records = _safe_get(analysis_json, "raw", "records", default=[]) or []
    snippets: List[str] = []

    for rec in raw_records[:max_records]:
        if isinstance(rec, dict):
            # Prefer key=value lines; stable ordering
            ordered_keys = ["mote_id", "moteid", "time", "ts", "epoch", "date", "temperature", "humidity", "light", "voltage", "value"]
            parts = []
            for k in ordered_keys:
                if k in rec and rec[k] is not None:
                    parts.append(f"{k}={rec[k]}")
            # Add remaining keys deterministically
            for k in sorted(rec.keys()):
                if k in ordered_keys:
                    continue
                v = rec.get(k)
                if v is None:
                    continue
                parts.append(f"{k}={v}")
            if parts:
                snippets.append(" ".join(parts))
            else:
                snippets.append(json.dumps(rec, ensure_ascii=False))
        else:
            snippets.append(str(rec))

    # If organizer produced an aggregate hint, add it explicitly
    agg = _safe_get(analysis_json, "context", "aggregate", default=None)
    if isinstance(agg, dict) and agg.get("value") is not None:
        snippets.insert(0, f"aggregate_value={agg.get('value')} field={agg.get('field')}")
    return snippets

def run_graph_with_debug(pipe, question: str) -> Tuple[str, List[str], Optional[str], Dict[str, Any]]:
    # Run pipeline and capture component outputs
    result = pipe.run(
        {"query_processor": {"query": question}, "generator": {"query": question}},
        include_outputs_from=["query_processor", "retriever", "organizer", "generator"],
    )

    cypher = _safe_get(result, "query_processor", "cypher", default=None)
    answer = _safe_get(result, "generator", "answer", default="")

    analysis_json = _safe_get(result, "organizer", "analysis_json", default={}) or {}
    judge_context = build_context_snippets_from_graph_analysis(analysis_json)

    debug = {
        "query_processor": {"cypher": cypher},
        "retriever": {"documents_total": len(_safe_get(result, "retriever", "documents", default=[]) or [])},
        "organizer": {"analysis_json": analysis_json},
        "generator": {"answer": answer, "model": GENERATOR_MODEL},
    }
    return str(answer), judge_context, cypher, debug


def run_base_with_debug(pipe, question: str) -> Tuple[str, List[str], Dict[str, Any]]:
    result = run_rag_pipeline(pipe, question, debug=False, preview_docs=0)
    answer = _extract_base_answer(result)

    # Evidence snippets for judge: use retrieved documents directly
    docs = None
    retr = result.get("retriever")
    if isinstance(retr, dict):
        docs = retr.get("documents")

    snippets: List[str] = []
    raw_docs: List[Dict[str, Any]] = []
    if isinstance(docs, list):
        for d in docs[:MAX_DEBUG_DOCS]:
            content = getattr(d, "content", None)
            meta = getattr(d, "meta", None)
            raw_docs.append({"content": content, "meta": meta})
            if content:
                snippets.append(str(content)[:1500])

    debug: Dict[str, Any] = {
        "retriever": {"documents_total": len(docs) if isinstance(docs, list) else None, "documents": raw_docs},
        "llm": {"answer": answer, "model": GENERATOR_MODEL},
    }
    return str(answer), snippets, debug


In [7]:
# -----------------------------
# Evaluation (both pipelines + comparison)
# (faithfulness removed — only EM and F1 retained)
# -----------------------------
def evaluate_both(
    *,
    graph_pipe,
    base_pipe,
    dataset_path: Path,
    max_samples: Optional[int] = None,
    show_progress: bool = True,
) -> Dict[str, Any]:
    data = load_dataset(dataset_path)
    if max_samples is not None:
        data = data[:max_samples]

    run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
    run_dir = OUTPUTS_DIR / run_id
    logs_dir = run_dir / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)


    rows: List[Dict[str, Any]] = []
    iterator = tqdm(data, disable=not show_progress, desc="Evaluating (both)")

    for idx, row in enumerate(iterator):
        q = str(row.get("question", row.get("query", "")))
        gold = str(row.get("gold_answer", ""))

        # --- Graph ---
        g_pred, g_ctx, g_cypher, g_dbg = "", [], None, {}
        g_err: Optional[str] = None
        try:
            g_pred, g_ctx, g_cypher, g_dbg = run_graph_with_debug(graph_pipe, q)
        except Exception as exc:
            g_err = str(exc)
            g_pred, g_ctx = f"[ERROR] {exc}", []
            g_dbg = {"error": g_err}

        g_cy_policy_ok, g_cy_policy_err = cypher_policy_check(g_cypher)
        g_cy_explain_ok, g_cy_explain_err = cypher_explain_ok(g_cypher)
        g_cy_err = g_cy_policy_err if not g_cy_policy_ok else g_cy_explain_err

        g_em = exact_match(g_pred, gold)
        g_f1 = f1_score(g_pred, gold)

        # --- Base ---
        b_pred, b_ctx, b_dbg = "", [], {}
        b_err: Optional[str] = None
        try:
            b_pred, b_ctx, b_dbg = run_base_with_debug(base_pipe, q)
        except Exception as exc:
            b_err = str(exc)
            b_pred, b_ctx = f"[ERROR] {exc}", []
            b_dbg = {"error": b_err}

        b_em = exact_match(b_pred, gold)
        b_f1 = f1_score(b_pred, gold)

        out_row: Dict[str, Any] = {
            "id": row.get("id"),
            "category": row.get("category"),
            "question": q,
            "gold_answer": gold,
            "graph": {
                "answer": g_pred,
                "exact_match": g_em,
                "f1": g_f1,
                "cypher": g_cypher,
                "cypher_policy_ok": g_cy_policy_ok,
                "cypher_explain_ok": g_cy_explain_ok,
                "cypher_error": g_cy_err,
                "error": g_err,
            },
            "base": {
                "answer": b_pred,
                "exact_match": b_em,
                "f1": b_f1,
                "error": b_err,
            },
            "meta": {
                "generator_model": GENERATOR_MODEL,
                "judge_model": JUDGE_MODEL,
            },
        }

        log_path = logs_dir / f"{idx:04d}_{row.get('id', 'sample')}.json"
        out_row["log_path"] = str(log_path)

        if SAVE_COMPONENT_OUTPUTS:
            out_row["component_outputs"] = {
                "graph": g_dbg,
                "base": b_dbg,
                "gold_row": row,
                "graph_judge_context": g_ctx,
                "base_judge_context": b_ctx,
            }
        save_json(log_path, out_row)

        rows.append(out_row)

    def _avg(values: List[Optional[float]]) -> float:
        vals = [v for v in values if isinstance(v, (int, float)) and v == v]
        return float(sum(vals) / len(vals)) if vals else float("nan")

    summary = {
        "run_id": run_id,
        "dataset_path": str(dataset_path),
        "n": len(rows),
        "graph": {
            "em": _avg([1.0 if r["graph"]["exact_match"] else 0.0 for r in rows]),
            "f1": _avg([r["graph"]["f1"] for r in rows]),
            "cypher_policy_ok_rate": _avg([1.0 if r["graph"]["cypher_policy_ok"] else 0.0 for r in rows]),
            "cypher_explain_ok_rate": _avg([1.0 if r["graph"]["cypher_explain_ok"] else 0.0 for r in rows]),
        },
        "base": {
            "em": _avg([1.0 if r["base"]["exact_match"] else 0.0 for r in rows]),
            "f1": _avg([r["base"]["f1"] for r in rows]),
        },
    }

    
    # -----------------------------
    # Persist outputs (simple + auditable)
    # -----------------------------
    results_path = run_dir / f"results_{run_id}.json"
    csv_path = run_dir / f"summary_{run_id}.csv"

    save_json(results_path, {"summary": summary, "rows": rows})

    # Flat CSV for quick comparison (one row per test case)
    csv_rows: List[Dict[str, Any]] = []
    for r in rows:
        csv_rows.append({
            "id": r.get("id"),
            "question": r.get("question"),
            "gold_answer": r.get("gold_answer"),
            "graph_answer": r.get("graph", {}).get("answer"),
            "graph_exact_match": r.get("graph", {}).get("exact_match"),
            "graph_f1": r.get("graph", {}).get("f1"),
            "graph_cypher_policy_ok": r.get("graph", {}).get("cypher_policy_ok"),
            "graph_cypher_explain_ok": r.get("graph", {}).get("cypher_explain_ok"),
            "graph_error": r.get("graph", {}).get("error"),
            "base_answer": r.get("base", {}).get("answer"),
            "base_exact_match": r.get("base", {}).get("exact_match"),
            "base_f1": r.get("base", {}).get("f1"),
            "base_error": r.get("base", {}).get("error"),
            "log_path": r.get("log_path"),
        })

    summary_csv_path = run_dir / f"summary_{run_id}.csv"
    save_csv(summary_csv_path, csv_rows)

    print("Saved results JSON to:", results_path)
    print("Saved summary CSV to:", summary_csv_path)
    print("Logs dir:", logs_dir)

    return {
        "run_id": run_id,
        "run_dir": str(run_dir),
        "results_path": str(results_path),
        "summary_csv_path": str(summary_csv_path),
        "summary": summary,
        "rows": rows,
    }


In [8]:

# -----------------------------
# Run
# -----------------------------

dataset_path = Path("/app/tests/datasets/eval_dataset_with_gold.jsonl")

if dataset_path is None:
    
    raise FileNotFoundError(f"Could not find eval_dataset_with_gold.jsonl")

results = evaluate_both(
    graph_pipe=graph_pipe,
    base_pipe=base_pipe,
    dataset_path=dataset_path,
    max_samples=MAX_SAMPLES,
    show_progress=SHOW_PROGRESS,
)
results["summary"]


Evaluating (both): 100%|██████████| 100/100 [33:49<00:00, 20.29s/it]

Saved results JSON to: /app/tests/outputs/20251222T100009/results_20251222T100009.json
Saved summary CSV to: /app/tests/outputs/20251222T100009/summary_20251222T100009.csv
Logs dir: /app/tests/outputs/20251222T100009/logs


{'run_id': '20251222T100009',
 'dataset_path': '/app/tests/datasets/eval_dataset_with_gold.jsonl',
 'n': 100,
 'graph': {'em': 0.1,
  'f1': 0.7264205731132214,
  'cypher_policy_ok_rate': 1.0,
  'cypher_explain_ok_rate': 1.0},
 'base': {'em': 0.0, 'f1': 0.1500213510135195}}